In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Imports & config
# ─────────────────────────────────────────────────────────────────────────────
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401  (registers 3d projection)

# ── Change this to your actual cache root ────────────────────────────────────
CACHE_ROOT = "./ModelNet10_mv_cache"   # contains train/ and test/
SPLIT      = "train"                   # "train" or "test"
# ─────────────────────────────────────────────────────────────────────────────

CLASSES     = ["bathtub","bed","chair","desk","dresser",
               "monitor","night_stand","sofa","table","toilet"]
VIEW_LABELS = ["Front (0°)", "Right (90°)", "Back (180°)",
               "Left (270°)", "Top (+89°)", "Bottom (−89°)"]



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Pick a random .pt file and load it
# ─────────────────────────────────────────────────────────────────────────────
all_files = sorted(Path(CACHE_ROOT, SPLIT).rglob("*.pt"))
assert all_files, f"No .pt files found under {CACHE_ROOT}/{SPLIT}"

pt_path = random.choice(all_files)
item    = torch.load(pt_path, map_location="cpu", weights_only=True)

rgb    = item["rgb"].float()   / 255.0   # (6, 3, H, W)  float32 [0,1]
depth  = item["depth"].float()           # (6, 1, H, W)  float32 [0,1]
voxels = item["voxels"].float()[0]       # (32, 32, 32)
label  = int(item["label"])
R      = voxels.shape[0]

print(f"File   : {pt_path.name}")
print(f"Class  : {CLASSES[label]}  (label={label})")
print(f"rgb    : {tuple(rgb.shape)}  range=[{rgb.min():.2f}, {rgb.max():.2f}]")
print(f"depth  : {tuple(depth.shape)}  range=[{depth.min():.2f}, {depth.max():.2f}]")
print(f"voxels : {tuple(voxels.shape)}  occupancy={voxels.mean()*100:.1f}%")



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Visualise: RGB views + depth maps + voxel slices + 3-D scatter
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 15))
fig.suptitle(
    f"Class: {CLASSES[label]}   |   {pt_path.name}",
    fontsize=14, y=0.99,
)

# ── Row 1: RGB (6 views) ──────────────────────────────────────────────────────
for v in range(6):
    ax = fig.add_subplot(4, 6, v + 1)
    ax.imshow(rgb[v].permute(1, 2, 0).numpy())
    ax.set_title(VIEW_LABELS[v], fontsize=8)
    ax.axis("off")

# ── Row 2: Depth maps ─────────────────────────────────────────────────────────
for v in range(6):
    ax = fig.add_subplot(4, 6, 6 + v + 1)
    im = ax.imshow(depth[v, 0].numpy(), cmap="plasma")
    ax.set_title(f"Depth  {VIEW_LABELS[v]}", fontsize=7)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# ── Row 3: Voxel cross-sections ───────────────────────────────────────────────
#   Left 3 cols → XY slices (top-down)
#   Right 3 cols → YZ slices (side-on)
slices = [R // 4, R // 2, 3 * R // 4]

for i, sl in enumerate(slices):
    ax = fig.add_subplot(4, 6, 12 + i + 1)
    ax.imshow(voxels[:, :, sl].numpy(), cmap="gray", vmin=0, vmax=1, origin="lower")
    ax.set_title(f"XY slice  z={sl}", fontsize=8)
    ax.axis("off")

for i, sl in enumerate(slices):
    ax = fig.add_subplot(4, 6, 12 + i + 4)
    ax.imshow(voxels[sl, :, :].numpy(), cmap="gray", vmin=0, vmax=1, origin="lower")
    ax.set_title(f"YZ slice  x={sl}", fontsize=8)
    ax.axis("off")

# ── Row 4: 3-D scatter of occupied voxels ─────────────────────────────────────
ax3d = fig.add_subplot(4, 6, (19, 24), projection="3d")
occ  = np.argwhere(voxels.numpy() > 0.5)   # (N, 3) — occupied voxel coords

if len(occ) > 0:
    # Colour by Z so depth is immediately readable
    sc = ax3d.scatter(
        occ[:, 0], occ[:, 1], occ[:, 2],
        c=occ[:, 2], cmap="viridis", s=3, alpha=0.6,
    )
    plt.colorbar(sc, ax=ax3d, fraction=0.03, pad=0.1, label="Z")
else:
    ax3d.text(0.5, 0.5, 0.5, "No occupied voxels", ha="center", transform=ax3d.transAxes)

ax3d.set_xlabel("X", fontsize=8)
ax3d.set_ylabel("Y", fontsize=8)
ax3d.set_zlabel("Z", fontsize=8)
ax3d.set_title(f"3D voxels  ({len(occ):,} occupied / {R**3:,} total)", fontsize=9)
ax3d.tick_params(labelsize=6)
ax3d.view_init(elev=25, azim=45)   # good default angle; drag to rotate in notebook

plt.tight_layout()
plt.show()


In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — (Optional) browse a specific class
# Change FILTER_CLASS to any of the 10 class names to load only from that class.
# ─────────────────────────────────────────────────────────────────────────────
FILTER_CLASS = "chair"   # ← change me

class_files = [f for f in all_files if f.parent.name == FILTER_CLASS]
print(f"Found {len(class_files)} files for class '{FILTER_CLASS}'")

if class_files:
    pt_path = random.choice(class_files)
    item    = torch.load(pt_path, map_location="cpu", weights_only=True)
    rgb     = item["rgb"].float() / 255.0
    depth   = item["depth"].float()
    voxels  = item["voxels"].float()[0]
    label   = int(item["label"])
    print(f"Loaded: {pt_path.name}  class={CLASSES[label]}")

    # Quick 6-panel RGB strip
    fig, axes = plt.subplots(1, 6, figsize=(20, 3))
    fig.suptitle(f"{CLASSES[label]}  —  {pt_path.name}", fontsize=12)
    for v in range(6):
        axes[v].imshow(rgb[v].permute(1, 2, 0).numpy())
        axes[v].set_title(VIEW_LABELS[v], fontsize=8)
        axes[v].axis("off")
    plt.tight_layout()
    plt.show()